In [ ]:
# @title 1) Student Info & Config
STUDENT_NAME = "Воропаев Фёдор Игоревич"  # @param {type:"string"}
GROUP = "11-301"         # @param {type:"string"}
ASSIGMENT = "CV_REAL_WORLD_PRACTICE"
SEED = 42
START_DATE = "2026-02-17"
DUE_DATE = "2026-03-03"


# HW12 — OCR / Document AI (Autograde)

Домашка сделана так, чтобы **работать без скачивания датасетов**: мы генерируем документ сами.

Баллы: **100**
- Task 1 (20): препроцессинг для OCR (`preprocess_for_ocr`)  
- Task 2 (20): CER и WER (`cer`, `wer`)  
- Task 3 (20): слова + bbox из OCR (`ocr_words_with_boxes`)  
- Task 4 (25): извлечение полей (`extract_fields`)  
- Task 5 (15): оценка полей (`score_fields`)  

Правила:
- Не редактируйте ячейки тестов и подсчёта баллов.
- Перед сдачей: Restart & Run all.


In [ ]:
# Install (Colab)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless pillow matplotlib


In [ ]:
import re, random
from datetime import datetime, timezone
from typing import Dict, Tuple, List, Any

import numpy as np
import cv2
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140

# ---------------- Reproducibility ----------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

# ---------------- Dates for last cell ----------------
def _parse_date(s: str):
    try:
        return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def _sec(td) -> float:
    return float(td.total_seconds())

start_dt = _parse_date(START_DATE)
due_dt = _parse_date(DUE_DATE)
submission_dt = datetime.now(timezone.utc)

# ---------------- Scoring ----------------
SCORES: Dict[str, float] = {}
def _set_score(task: str, pts: float):
    SCORES[task] = float(pts)
def _total_score():
    return float(sum(SCORES.values()))
def _print_scores():
    print("Scores:")
    for k in sorted(SCORES.keys()):
        print(f"  {k}: {SCORES[k]:.1f}")
    print("TOTAL:", _total_score())
def _assert(cond: bool, msg: str):
    if not cond:
        raise AssertionError(msg)

# OCR helpers
def ocr_text(img: Image.Image, lang="eng") -> str:
    config = "--oem 3 --psm 6"
    return pytesseract.image_to_string(img, lang=lang, config=config)


## Данные: синтетический инвойс

Мы генерируем один документ. Он используется и в заданиях, и в тестах.


In [ ]:
def make_invoice(seed=2, angle_deg=0.0, noise=0.0, low_contrast=False) -> Tuple[Image.Image, Dict[str,str]]:
    rng = np.random.RandomState(seed)
    W, H = 900, 520
    bg = 245 if not low_contrast else 220
    fg = 10  if not low_contrast else 60

    img = Image.new("L", (W, H), color=bg)
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()

    invoice_no = f"INV-{rng.randint(10000, 99999)}"
    date = f"2026-0{rng.randint(2, 9)}-{rng.randint(10, 28)}"
    total = f"{rng.randint(12, 199)}.{rng.randint(0, 99):02d}"

    lines = [
        "INVOICE",
        f"Invoice No: {invoice_no}",
        f"Date: {date}",
        "",
        "Items:",
        "  1) Service A    49.90",
        "  2) Service B    19.50",
        "  3) Product C    12.00",
        "",
        f"TOTAL: {total}",
        "Thank you!",
    ]

    y = 30
    for line in lines:
        draw.text((40, y), line, fill=fg, font=font)
        y += 24

    draw.rectangle([35, y-48, 420, y-20], outline=fg, width=1)

    if abs(angle_deg) > 0.001:
        img = img.rotate(angle_deg, expand=True, fillcolor=bg)

    if noise > 0:
        arr = np.array(img).astype(np.float32)
        arr += rng.normal(0, noise, size=arr.shape)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr, mode="L")

    gt = {"invoice_no": invoice_no, "date": date, "total": total}
    return img, gt

DOC_IMG, GT = make_invoice(seed=2, angle_deg=4.0, noise=10.0, low_contrast=True)
print("GT:", GT)


## Task 1 (20 pts) — Препроцессинг для OCR

Реализуйте `preprocess_for_ocr(img)`:
- вход: PIL Image (L или RGB)
- выход: PIL Image (желательно 'L')
- цель: улучшить читаемость (обычно binarization + denoise)

Тест проверит:
- что функция возвращает PIL Image
- что результат детерминированный
- что после preprocessing OCR-качество не хуже, чем без него (по CER на ключевых строках)


In [ ]:
# TODO: implement
def preprocess_for_ocr(img: Image.Image) -> Image.Image:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 1
try:
    img = DOC_IMG
    out = preprocess_for_ocr(img)
    _assert(isinstance(out, Image.Image), "preprocess_for_ocr must return PIL.Image")
    out2 = preprocess_for_ocr(img)
    _assert(np.array_equal(np.array(out), np.array(out2)), "preprocess_for_ocr must be deterministic for same input")

    raw = ocr_text(img)
    pre = ocr_text(out)

    # key line: invoice no and date should become more readable or equal
    ref = f"Invoice No: {GT['invoice_no']} Date: {GT['date']} TOTAL: {GT['total']}"
    # CER should not get worse by more than a tiny tolerance (OCR is noisy)
    def _edit_distance(a, b):
        n, m = len(a), len(b)
        dp = np.zeros((n+1, m+1), dtype=np.int32)
        dp[:,0] = np.arange(n+1); dp[0,:] = np.arange(m+1)
        for i in range(1, n+1):
            for j in range(1, m+1):
                cost = 0 if a[i-1] == b[j-1] else 1
                dp[i,j] = min(dp[i-1,j] + 1, dp[i,j-1] + 1, dp[i-1,j-1] + cost)
        return int(dp[n,m])
    def _cer(r, h):
        return 0.0 if len(r)==0 else _edit_distance(r, h)/len(r)

    cer_raw = _cer(ref, raw.replace("\n"," "))
    cer_pre = _cer(ref, pre.replace("\n"," "))
    print("CER raw:", round(cer_raw,4), "CER pre:", round(cer_pre,4))
    _assert(cer_pre <= cer_raw + 0.02, "Preprocessing should not significantly worsen CER")
    _set_score("task1", 20)
    print("✅ Task 1 passed")
except Exception as e:
    print("❌ Task 1 failed:", e)
    _set_score("task1", 0)


## Task 2 (20 pts) — CER и WER

Реализуйте:
- `cer(ref, hyp)` — Character Error Rate
- `wer(ref, hyp)` — Word Error Rate

Используйте Levenshtein edit distance.


In [ ]:
# TODO: implement
def edit_distance(a, b) -> int:
    # YOUR CODE HERE
    raise NotImplementedError()

def cer(ref: str, hyp: str) -> float:
    # YOUR CODE HERE
    raise NotImplementedError()

def wer(ref: str, hyp: str) -> float:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 2
try:
    _assert(edit_distance("kitten", "sitting") == 3, "edit_distance must be 3 for kitten->sitting")
    _assert(abs(cer("abcd", "abxd") - 0.25) < 1e-9, "CER incorrect")
    _assert(abs(wer("a b c", "a x c") - (1/3)) < 1e-9, "WER incorrect")
    _set_score("task2", 20)
    print("✅ Task 2 passed")
except Exception as e:
    print("❌ Task 2 failed:", e)
    _set_score("task2", 0)


## Task 3 (20 pts) — Слова + bbox из OCR (детекция слов)

Реализуйте `ocr_words_with_boxes(img)`:
- возвращает список словарей:
  - `text`: str
  - `bbox`: (x1,y1,x2,y2) ints
  - `conf`: float
- используйте `pytesseract.image_to_data(..., output_type=DICT)`


In [ ]:
# TODO: implement
def ocr_words_with_boxes(img: Image.Image, lang="eng") -> List[Dict[str, Any]]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 3
try:
    out = ocr_words_with_boxes(preprocess_for_ocr(DOC_IMG))
    _assert(isinstance(out, list), "must return list")
    _assert(len(out) >= 5, "expected at least 5 recognized words")
    w = out[0]
    _assert("text" in w and "bbox" in w and "conf" in w, "dict must have text/bbox/conf")
    x1,y1,x2,y2 = w["bbox"]
    _assert(all(isinstance(v, int) for v in [x1,y1,x2,y2]), "bbox must be ints")
    _assert(x2 >= x1 and y2 >= y1, "bbox must be valid")
    _set_score("task3", 20)
    print("✅ Task 3 passed")
except Exception as e:
    print("❌ Task 3 failed:", e)
    _set_score("task3", 0)


## Task 4 (25 pts) — Извлечение полей (Document AI)

Реализуйте:
- `normalize_date(s)` → 'YYYY-MM-DD' или ''  
- `normalize_money(s)` → 'NNN.NN' или ''  
- `extract_fields(text)` → {'invoice_no','date','total'}

Тест допускает мелкие OCR-ошибки, но ожидает корректную нормализацию и регулярные выражения.


In [ ]:
# TODO: implement
def normalize_date(s: str) -> str:
    # YOUR CODE HERE
    raise NotImplementedError()

def normalize_money(s: str) -> str:
    # YOUR CODE HERE
    raise NotImplementedError()

def extract_fields(text: str) -> Dict[str, str]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 4
try:
    txt = ocr_text(preprocess_for_ocr(DOC_IMG)).replace("\n", " ")
    f = extract_fields(txt)
    _assert(set(f.keys()) == {"invoice_no","date","total"}, "extract_fields must return invoice_no/date/total keys")

    # normalize expected formats
    _assert(re.match(r"^INV-\d{5}$", f["invoice_no"]) is not None or f["invoice_no"] == "", "invoice_no format should be INV-12345")
    _assert(re.match(r"^20\d{2}-\d{2}-\d{2}$", f["date"]) is not None or f["date"] == "", "date must be YYYY-MM-DD")
    _assert(re.match(r"^\d+\.\d{2}$", f["total"]) is not None or f["total"] == "", "total must be NNN.NN")

    # Should extract at least 2/3 fields correctly most of the time
    gt = {"invoice_no": GT["invoice_no"], "date": GT["date"], "total": f"{float(GT['total']):.2f}"}
    hit = 0
    hit += int(f["invoice_no"] == gt["invoice_no"])
    hit += int(f["date"] == gt["date"])
    hit += int(f["total"] == gt["total"])
    print("Extracted:", f)
    print("GT norm:", gt)
    _assert(hit >= 2, "Expected at least 2/3 fields correct on synthetic doc")
    _set_score("task4", 25)
    print("✅ Task 4 passed")
except Exception as e:
    print("❌ Task 4 failed:", e)
    _set_score("task4", 0)


## Task 5 (15 pts) — Оценка качества извлечения полей

Реализуйте `score_fields(gt, pred)`:
- возвращает `accuracy` (доля совпавших полей из 3)
- возвращает `details` (словарь по полям: True/False)

Формат:
`{"accuracy": float, "details": {"invoice_no": bool, "date": bool, "total": bool}}`


In [ ]:
# TODO: implement
def score_fields(gt: Dict[str,str], pred: Dict[str,str]) -> Dict[str, Any]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 5
try:
    gt = {"invoice_no":"INV-12345", "date":"2026-02-17", "total":"19.50"}
    pr = {"invoice_no":"INV-12345", "date":"2026-02-17", "total":"19.40"}
    s = score_fields(gt, pr)
    _assert("accuracy" in s and "details" in s, "Missing keys")
    _assert(abs(float(s["accuracy"]) - (2/3)) < 1e-9, "accuracy must be 2/3")
    _assert(s["details"]["total"] is False and s["details"]["invoice_no"] is True, "details mismatch")
    _set_score("task5", 15)
    print("✅ Task 5 passed")
except Exception as e:
    print("❌ Task 5 failed:", e)
    _set_score("task5", 0)


In [ ]:
# FINAL: total points
total = _total_score()
_print_scores()
print("\nTOTAL POINTS / 100 =", total)


In [ ]:
def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)
	
# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
final_score = max(0.0, total * (1.0 - min(1.0, pf)))
print(final_score)
import json
final = {
    "name": STUDENT_NAME,
    "group": GROUP,
    "assignment":ASSIGMENT,
    "score": float(total),
    "penalty_score": float(final_score),
    "due_date": DUE_DATE,
    "start_date": START_DATE,
}
print(json.dumps(final, ensure_ascii=False))
